In [ ]:
# ════════════════════════════════════════════════════════════════════
# Stage 2 — Assessment LoRA Adapter (Gemma 3 12B — Primary)
# ════════════════════════════════════════════════════════════════════
# PIPELINE STAGES:
#   Stage 1a = Relevance, Priority, Type (3 LoRA passes per email)
#   Stage 1b = Identifying ideal vs actual context and evidence
#   Stage 2a = Complexity and urgency
#   Stage 2b = Research, recommendations and strategy
#
# THIS NOTEBOOK: Trains the PRIMARY assessment adapter (Gemma 3 12B).
#   Produces structured JSON with 9 fields:
#   complexity, urgency, topic_esg, topic_operational, entity_routing,
#   sentiment, formality, response_type, reasoning
#
# AI CASCADE (3-brain):
#   1. Gemma 3 12B LoRA (THIS) — primary assessor
#   2. Mistral 7B LoRA — secondary/multilingual fallback
#   3. Claude API — escalation for high-complexity
#
# Model: @cf/google/gemma-3-12b-it (gated — needs HF login)
# Adapter worker: servers/mcp/network-http/cloudflare/ai/cf-adapters/
# Email worker: servers/mcp/network-http/cloudflare/email/cf-email/
# Verify endpoint: /api/assess
# Checklist: notebooks/checklists/stage-2b-assess/tools/ai-cascade.json
# Schema: schemas/mobicycle-{account}/pipelines/email/email_enrichment.json
#
# FIELD MAPPING — notebook output → enrichment schema:
#   complexity         → complexity (low | medium | high)
#   urgency            → urgency (immediate | time-sensitive | standard | no-deadline)
#   response_type      → recommendation_type (acknowledge | respond | escalate | file | forward)
#   reasoning          → complexity_reasoning + urgency_reasoning (split by Worker)
#   topic_esg          → research_context.topic_esg (nested in JSON field)
#   topic_operational  → research_context.topic_operational (nested in JSON field)
#   entity_routing     → research_context.entity_routing (nested in JSON field)
#   sentiment          → research_context.sentiment (nested in JSON field)
#   formality          → research_context.formality (nested in JSON field)
#
# SCHEMA GAPS — fields the adapter does NOT produce (filled by Worker logic):
#   complexity_sources, urgency_sources — populated from RAG retrieval metadata
#   recommendation_outline, recommendation_reasoning — requires separate prompt
#   deadline, deadline_source — extracted from email body + key_dates
#   escalation_required, escalation_to, escalation_reason — derived from complexity + type
#   strategy_* fields — separate strategy prompt after recommendation
#   draft_* fields — generated in strategy step, not assessment
# Run once per account. Change ACCOUNT below.
#
# RULES (from notebooks/platforms/errors-do-not-delete/):
#   1. NEVER delete or overwrite existing comments — especially Cell 3 descriptions.
#   2. Before editing, verify every cell achieves the purpose stated above.
#   3. Colab MCP tools may not register. Fall back to editing .ipynb files locally.
#   4. Do NOT call open_colab_browser_connection if a notebook is already open.
#
# WARNING: This notebook gets overwritten when regenerated.
# Once trained and verified, push to GitHub immediately:
#   Training repos: org-mobicycle/training, org-mobicycle-ee/training, etc.
#   NEVER push to MobiCycleLtd/technologies.
# ════════════════════════════════════════════════════════════════════
# ── Cell 0: Config ──
ACCOUNT = "mobicycle.consulting"  # Change per account
TASK = "assess"  # Stage 2 assessment (NOT Stage 1 classification)
MODEL = "google/gemma-3-12b-it"  # Gemma 3 12B (80K context, multilingual, LoRA on CF)
MODEL_TYPE = "gemma3"  # Used in adapter_config.json for Workers AI
WORKERS_AI_MODEL = "@cf/google/gemma-3-12b-it"  # Cloudflare inference endpoint
ADAPTER_PREFIX = "assess-gemma3"  # Adapter name: assess-gemma3-{account}
EPOCHS = 3
BATCH_SIZE = 1  # Reduced for 12B on T4 (15GB VRAM)
LEARNING_RATE = 1e-4  # Lower LR for larger model
BLOCK_SIZE = 2048  # Gemma 3 supports 80K but we limit for VRAM
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
GRADIENT_ACCUMULATION = 16  # Effective batch = 16 (compensates for batch_size=1)

import json
_cfg = {k: v for k, v in locals().items() if k.isupper()}
with open('/content/config.json', 'w') as f:
    json.dump(_cfg, f)
print(f'Account: {ACCOUNT}')
print(f'Task: {TASK} (Stage 2 assessment)')
print(f'Model: {MODEL}')
print(f'Workers AI: {WORKERS_AI_MODEL}')
print(f'Adapter: {ADAPTER_PREFIX}-{ACCOUNT}')
print(f'Block size: {BLOCK_SIZE}')
print(f'Batch: {BATCH_SIZE} x {GRADIENT_ACCUMULATION} grad accum = {BATCH_SIZE * GRADIENT_ACCUMULATION} effective')
print('Config saved.')


In [ ]:
# ── Cell 1: Install ──
!pip install -q torch transformers accelerate peft bitsandbytes datasets trl huggingface_hub


In [ ]:
# ── Cell 1.5: Log in to Hugging Face ──
# Gemma 3 is gated — accept license at: https://huggingface.co/google/gemma-3-12b-it
# Add your HF token as a Colab secret named HF_TOKEN (key icon in left sidebar)
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Successfully logged in to Hugging Face.')
except Exception as e:
    print(f'Failed to log in. Ensure HF_TOKEN secret is set: {e}')


In [ ]:
# ── Cell 2: GPU Check ──
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > T4 GPU')
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} ({mem:.1f} GB)')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# STAGE 2 ASSESSMENT — STRUCTURED JSON OUTPUT
# ════════════════════════════════════════════════════════════════════
# This adapter produces structured JSON assessments for emails that
# have ALREADY been classified by Stage 1 (relevance + priority).
#
# Output fields:
#   complexity:       low | medium | high
#   urgency:          immediate | time-sensitive | standard | no-deadline
#   topic_esg:        scope-3 | biodiversity | pollution | null
#   topic_operational: legal | financial | client | infrastructure | hr | partnership
#   entity_routing:   same_account | mobicycle.{division}
#   sentiment:        positive | neutral | negative | hostile
#   formality:        formal | business | casual
#   response_type:    acknowledge | respond | escalate | file | forward
#   reasoning:        1-2 sentence explanation
#
# NOT the same as Stage 1 LoRA adapters (relevance/priority).
# ════════════════════════════════════════════════════════════════════
# ── Cell 3: Training Data (120 Assessment Examples) ──
import csv, os, random, json

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

# Format: (from_name, from_email, subject, body_snippet, assessment_json)
EXAMPLES = [
    # ──── HIGH COMPLEXITY (40) ────
    ("Crown Court Listing", "listing@courts.gsi.gov.uk",
     "Hearing listed: Case MC-2025-0047",
     "Your case MC-2025-0047 has been listed for hearing on 15 April 2026 at 10:00 before HHJ Thompson. Skeleton arguments must be filed by 8 April. Failure to comply may result in sanctions.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"legal","entity_routing":"same_account","sentiment":"neutral","formality":"formal","response_type":"escalate","reasoning":"Court hearing with filing deadline. Requires skeleton argument preparation and legal strategy. High stakes — sanctions for non-compliance."}),

    ("Environment Agency", "enforcement@environment-agency.gov.uk",
     "Compliance notice ref EA/2026/4412",
     "We have identified potential breaches of your environmental permit EP/L1234. You are required to provide a written response within 14 days setting out remedial actions taken or proposed.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":"pollution","topic_operational":"legal","entity_routing":"same_account","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Regulatory enforcement notice with 14-day response deadline. Pollution compliance breach. Requires legal review before responding."}),

    ("ICO", "enforcement@ico.org.uk",
     "Data breach notification: response required within 72 hours",
     "We have received a report of a personal data breach affecting your organisation. Under Article 33 GDPR, you must provide a full report within 72 hours of becoming aware. Please complete the enclosed form.",
     {"complexity":"high","urgency":"immediate","topic_esg":None,"topic_operational":"legal","entity_routing":"mobicycle.tech","sentiment":"neutral","formality":"formal","response_type":"escalate","reasoning":"GDPR data breach with 72-hour statutory deadline. Requires tech division input on breach scope. Regulatory response with legal liability."}),

    ("Opposing Counsel", "litigation@lawfirm.co.uk",
     "Without prejudice: settlement proposal",
     "Further to the ongoing proceedings in case MC-2025-0047, our client instructs us to make the following settlement offer on a without prejudice basis. The offer is open for acceptance for 14 days.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"legal","entity_routing":"same_account","sentiment":"neutral","formality":"formal","response_type":"escalate","reasoning":"Settlement offer with 14-day window. Without prejudice — requires legal strategy assessment. Financial and legal implications of acceptance or rejection."}),

    ("HMRC", "penalties@hmrc.gov.uk",
     "Corporation tax penalty: appeal deadline 30 April",
     "A penalty of £1,250 has been issued for late filing of Corporation Tax return CT600 for period ending 31 March 2025. You may appeal within 30 days of this notice.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"financial","entity_routing":"mobicycle.capital","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Tax penalty with 30-day appeal window. Financial impact £1,250. Requires accountant review and potential appeal preparation."}),

    ("FCA", "enforcement@fca.org.uk",
     "Information requirement under s.165 FSMA 2000",
     "The Financial Conduct Authority requires you to provide the following information under section 165 of the Financial Services and Markets Act 2000. Failure to comply is a criminal offence.",
     {"complexity":"high","urgency":"immediate","topic_esg":None,"topic_operational":"legal","entity_routing":"mobicycle.capital","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Statutory information requirement from financial regulator. Criminal offence for non-compliance. Requires immediate legal counsel and capital division coordination."}),

    ("Employment Tribunal", "listing@employmenttribunals.gov.uk",
     "Notice of preliminary hearing: unfair dismissal claim",
     "A preliminary hearing has been listed for 22 April 2026. The claimant alleges unfair dismissal and disability discrimination. Please confirm attendance and provide your response.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"legal","entity_routing":"same_account","sentiment":"neutral","formality":"formal","response_type":"escalate","reasoning":"Employment tribunal hearing with discrimination claim. Multi-faceted legal defence needed. Reputational and financial risk."}),

    ("Client CEO", "ceo@majorretailer.com",
     "Urgent: board presentation on supply chain emissions tomorrow",
     "I need the scope 3 analysis for our board meeting tomorrow at 9am. The non-exec directors are asking tough questions about our SBTi targets. Can you join the call to present?",
     {"complexity":"high","urgency":"immediate","topic_esg":"scope-3","topic_operational":"client","entity_routing":"same_account","sentiment":"negative","formality":"business","response_type":"respond","reasoning":"Client CEO requesting board-level presentation by tomorrow. Scope 3 SBTi analysis. High client relationship stakes. Time pressure is extreme."}),

    ("Solicitor", "dispute@clientlaw.co.uk",
     "Multi-party dispute: mediation proposal",
     "Three parties are now involved in the waste management contract dispute. The court has suggested mediation before proceeding to trial. Costs to date exceed £85,000. We recommend settling.",
     {"complexity":"high","urgency":"standard","topic_esg":"pollution","topic_operational":"legal","entity_routing":"same_account","sentiment":"neutral","formality":"formal","response_type":"escalate","reasoning":"Multi-party legal dispute with significant costs. Mediation decision requires strategic judgement. Waste management contract involves pollution ESG area."}),

    ("Bank Compliance", "compliance@businessbank.co.uk",
     "Account freeze: KYC documents required immediately",
     "Your business account has been temporarily restricted pending completion of our enhanced due diligence review. Please provide updated UBO declarations, source of funds evidence, and three years of audited accounts.",
     {"complexity":"high","urgency":"immediate","topic_esg":None,"topic_operational":"financial","entity_routing":"mobicycle.capital","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Bank account frozen. Business operations impacted. Requires extensive document gathering across divisions. Capital division must coordinate."}),

    ("SFO", "enquiries@sfo.gov.uk",
     "Document preservation notice",
     "You are required to preserve all documents, communications, and electronic records relating to contracts with [REDACTED] from 2023 to present. Failure to preserve may constitute obstruction.",
     {"complexity":"high","urgency":"immediate","topic_esg":None,"topic_operational":"legal","entity_routing":"same_account","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Serious Fraud Office preservation notice. Criminal investigation context. Must implement litigation hold immediately across all systems."}),

    ("Kohus EE", "info@kohus.ee",
     "Kohtumäärus: dokumendid esitada 10 päeva jooksul",
     "Harju Maakohus nõuab dokumentide esitamist tsiviilasjas nr 2-26-1234. Esitamise tähtaeg on 10 kalendripäeva käesoleva määruse kättetoimetamisest.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"legal","entity_routing":"mobicycle.ee","sentiment":"neutral","formality":"formal","response_type":"escalate","reasoning":"Estonian court order requiring document submission within 10 days. Must route to Estonian entity. Legal compliance with strict deadline."}),

    ("DEFRA", "enforcement@defra.gov.uk",
     "Extended Producer Responsibility: non-compliance investigation",
     "Our investigation indicates potential non-compliance with the Packaging Extended Producer Responsibility regulations. A formal interview under caution is being arranged. You are entitled to legal representation.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":"pollution","topic_operational":"legal","entity_routing":"same_account","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Regulatory investigation with interview under caution. EPR compliance issue in pollution/circular economy area. Criminal proceedings possible."}),

    ("EU Commission", "infringements@ec.europa.eu",
     "CSRD reporting: formal inquiry on sustainability disclosures",
     "Under the Corporate Sustainability Reporting Directive, we are conducting a review of your group's sustainability disclosures for FY2025. Please provide all supporting documentation within 30 days.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":"scope-3","topic_operational":"legal","entity_routing":"mobicycle.eu","sentiment":"neutral","formality":"formal","response_type":"escalate","reasoning":"EU Commission formal inquiry on CSRD compliance. Cross-divisional data gathering needed. Route to EU entity. 30-day statutory deadline."}),

    ("Insurance", "claims@hiscox.com",
     "PI claim notification: client alleges negligent advice",
     "Your client RetailCo Ltd has notified us of a potential claim alleging negligent ESG consulting advice leading to regulatory fines of approximately £450,000. Please provide your file notes and all correspondence.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"legal","entity_routing":"same_account","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Professional indemnity claim with significant quantum. Reputational risk. Requires complete file review and legal strategy."}),

    ("Landlord Solicitor", "property@commerciallaw.co.uk",
     "Lease forfeiture: breach of covenant",
     "Our client intends to exercise their right of forfeiture under clause 4.2 of the lease due to persistent breach of the repair covenant. You have 14 days to remedy the breach.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"legal","entity_routing":"same_account","sentiment":"hostile","formality":"formal","response_type":"escalate","reasoning":"Lease forfeiture threat. Loss of office premises if not addressed. 14-day remedy period. Requires property solicitor and business continuity planning."}),

    ("Whistleblower", "anonymous@protonmail.com",
     "Confidential: concerns about waste disposal practices",
     "I work for your waste contractor and want to report that hazardous e-waste is being illegally dumped rather than properly recycled. I have photographic evidence.",
     {"complexity":"high","urgency":"immediate","topic_esg":"pollution","topic_operational":"legal","entity_routing":"same_account","sentiment":"neutral","formality":"casual","response_type":"escalate","reasoning":"Whistleblower report of illegal waste dumping. Environmental crime. Must preserve evidence, assess supply chain exposure, potentially notify regulators."}),

    ("M&A Advisory", "deals@investmentbank.com",
     "Confidential: acquisition approach for your games division",
     "Our client, a major educational technology company, wishes to explore acquiring MobiCycle Games Ltd. They are prepared to offer indicative terms in the range of £2-3M. This approach is strictly confidential.",
     {"complexity":"high","urgency":"standard","topic_esg":None,"topic_operational":"financial","entity_routing":"mobicycle.capital","sentiment":"positive","formality":"formal","response_type":"escalate","reasoning":"Acquisition approach requiring board-level decision. Confidential. Financial and strategic implications. Route to capital division."}),

    ("Tax Authority EE", "maksuamet@emta.ee",
     "Maksukontroll: OÜ MobiCycle käibemaksu deklaratsioon",
     "Maksu- ja Tolliamet teostab OÜ MobiCycle käibemaksu kontrolli perioodi 2025 Q3-Q4. Palume esitada kõik arved ja raamatupidamisdokumendid 14 päeva jooksul.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"financial","entity_routing":"mobicycle.ee","sentiment":"neutral","formality":"formal","response_type":"escalate","reasoning":"Estonian tax authority VAT audit. 14-day deadline for document submission. Route to Estonian entity. Requires accountant coordination."}),

    ("Joint Venture Partner", "legal@partnerfirm.com",
     "Dispute notice: breach of JV agreement clause 8.3",
     "We believe MobiCycle Consulting has breached clause 8.3 of our Joint Venture Agreement by engaging in competing work with GreenTech Ltd. We reserve all rights and invite your response within 21 days.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"legal","entity_routing":"same_account","sentiment":"hostile","formality":"formal","response_type":"escalate","reasoning":"JV dispute with potential breach of non-compete. Partnership relationship at stake. Legal review of agreement terms required. 21-day response window."}),

    ("Client Legal", "legal@energyco.com",
     "Indemnity claim under consulting agreement",
     "Under clause 12 of our consulting agreement, we are making a formal indemnity claim for losses arising from errors in the carbon footprint calculation you provided. Losses are estimated at £280,000.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":"scope-3","topic_operational":"legal","entity_routing":"same_account","sentiment":"hostile","formality":"formal","response_type":"escalate","reasoning":"Indemnity claim for allegedly negligent carbon calculations. £280K exposure. Scope 3 ESG area. Must notify PI insurers and engage solicitors."}),

    ("Data Subject", "privacy@individual.com",
     "Subject Access Request under UK GDPR",
     "Under Article 15 of the UK GDPR, I request a copy of all personal data you hold about me. I also request details of any third parties with whom you have shared my data. You have one calendar month to respond.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"legal","entity_routing":"mobicycle.tech","sentiment":"neutral","formality":"formal","response_type":"respond","reasoning":"GDPR SAR with statutory 1-month deadline. Requires data search across all systems. Tech division needed to extract data. Legal compliance obligation."}),

    ("Pension Regulator", "compliance@tpr.gov.uk",
     "Contribution notice: employer obligations",
     "We are issuing a contribution notice under section 38 of the Pensions Act 2004. Your company has failed to meet its auto-enrolment obligations. A penalty may be imposed.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"financial","entity_routing":"same_account","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Pension regulator enforcement. Statutory non-compliance. Financial penalty risk. Requires HR and finance coordination to demonstrate compliance."}),

    ("Board Member", "ned@boardmember.com",
     "Concerns about company direction and ESG strategy",
     "I've been reviewing the Q4 financials alongside our ESG commitments. I have serious concerns about the sustainability of our current growth strategy. I'd like to raise a formal board-level discussion.",
     {"complexity":"high","urgency":"standard","topic_esg":"scope-3","topic_operational":"financial","entity_routing":"same_account","sentiment":"negative","formality":"business","response_type":"escalate","reasoning":"Board member raising strategic concerns. Intersects ESG and financial strategy. Requires careful, considered response. Governance implications."}),

    ("HSE Inspector", "inspections@hse.gov.uk",
     "Improvement notice: workplace safety violations",
     "Following our inspection on 12 March, we are issuing an improvement notice under section 21 HSWA 1974. You must implement the specified improvements within 21 days or face prosecution.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"legal","entity_routing":"same_account","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"HSE improvement notice with 21-day compliance deadline. Prosecution risk. Requires safety remediation plan and legal review."}),

    ("Creditor", "recovery@debtcollection.com",
     "Final demand: outstanding invoice £47,500",
     "Despite multiple reminders, invoice INV-2025-4421 for £47,500 remains unpaid. If payment is not received within 7 days, we will instruct solicitors to issue proceedings without further notice.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"financial","entity_routing":"mobicycle.capital","sentiment":"hostile","formality":"formal","response_type":"escalate","reasoning":"Threatened legal action over £47.5K debt. 7-day deadline. Financial and reputational risk. Capital division must review payment status."}),

    ("EU TNFD Secretariat", "compliance@tnfd.global",
     "Mandatory biodiversity disclosure: non-compliance warning",
     "Your organisation has failed to submit the required TNFD-aligned biodiversity impact assessment for FY2025. Under the EU Nature Restoration Law, non-compliance may result in enforcement action.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":"biodiversity","topic_operational":"legal","entity_routing":"mobicycle.eu","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"TNFD biodiversity disclosure non-compliance. EU regulatory enforcement risk. Route to EU entity. Requires cross-divisional data gathering."}),

    ("Planning Inspector", "hearings@planninginspectorate.gov.uk",
     "Public inquiry: expert witness attendance required",
     "You are required to attend as an expert witness at the public inquiry into the proposed waste processing facility. Your environmental impact assessment will be examined in detail.",
     {"complexity":"high","urgency":"standard","topic_esg":"pollution","topic_operational":"legal","entity_routing":"same_account","sentiment":"neutral","formality":"formal","response_type":"respond","reasoning":"Expert witness requirement at public inquiry. Pollution/waste facility. Professional reputation at stake. Requires preparation and review of EIA."}),

    ("Auditor", "audit@big4firm.com",
     "Material misstatement: ESG metrics in annual report",
     "During our audit of FY2025, we have identified a potential material misstatement in the reported Scope 3 emissions figures. The variance is approximately 340% from our independent calculations.",
     {"complexity":"high","urgency":"immediate","topic_esg":"scope-3","topic_operational":"financial","entity_routing":"same_account","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Material misstatement in published ESG data. 340% variance in Scope 3 figures. Reputational, regulatory, and legal exposure. Requires immediate investigation."}),

    ("WEEE Regulator", "producer-compliance@ofgem.gov.uk",
     "WEEE producer registration: compliance failure",
     "Your organisation has not met its WEEE collection obligations for Q4 2025. Under the WEEE Regulations 2013, this is an offence. A compliance plan must be submitted within 28 days.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":"pollution","topic_operational":"legal","entity_routing":"same_account","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"WEEE compliance failure with 28-day deadline. E-waste regulation core to MobiCycle mission. Criminal offence. Requires compliance plan."}),

    ("Investor", "esg@pensionscheme.com",
     "ESG due diligence: concerns about greenwashing allegations",
     "Press reports have raised concerns about the accuracy of ESG claims made by companies in our portfolio, including MobiCycle. We require a formal response addressing each allegation before our next investment committee.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":"scope-3","topic_operational":"financial","entity_routing":"mobicycle.capital","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Investor due diligence on greenwashing allegations. Investment relationship at risk. Requires coordinated response across ESG and capital divisions."}),

    ("Union Rep", "rep@uniteunion.org",
     "Collective grievance: health and safety concerns",
     "On behalf of 12 members at your processing facility, we are raising a formal collective grievance regarding exposure to hazardous materials without adequate PPE. We expect a formal response within 5 working days.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":"pollution","topic_operational":"hr","entity_routing":"same_account","sentiment":"hostile","formality":"formal","response_type":"escalate","reasoning":"Collective grievance about hazardous material exposure. H&S and employment law crossover. 5-day response expected. Potential tribunal and HSE involvement."}),

    ("Carbon Trust", "verification@carbontrust.com",
     "Verification failure: Product Carbon Footprint certification",
     "Your PCF certification for the MobiCycle platform has failed verification due to significant errors in the Scope 3 upstream calculations. The certification will be withdrawn unless corrected within 30 days.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":"scope-3","topic_operational":"client","entity_routing":"same_account","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Carbon certification failure. Core business credibility at stake. Scope 3 calculation errors. 30-day correction window. Client-facing impact."}),

    ("Client Procurement", "procurement@retailgroup.com",
     "Contract termination notice: underperformance",
     "Under clause 15.2 of our master services agreement, we are providing 60 days' notice of termination due to persistent underperformance in the agreed KPIs for the sustainability consulting engagement.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"client","entity_routing":"same_account","sentiment":"hostile","formality":"formal","response_type":"escalate","reasoning":"Major client terminating contract. Revenue loss and reputational damage. 60-day window to remedy or negotiate. Requires senior management attention."}),

    ("Grant Body", "compliance@innovateuk.ukri.org",
     "Grant clawback: failure to meet milestones",
     "Our review has determined that MobiCycle has not met milestones 3 and 4 of grant agreement IUK-2025-0789. We are initiating clawback proceedings for the £175,000 already disbursed.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":None,"topic_operational":"financial","entity_routing":"mobicycle.capital","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Grant clawback of £175K. Financial and reputational impact. Must demonstrate milestone achievement or negotiate. Capital division involvement."}),

    ("Media", "investigations@guardian.com",
     "Press enquiry: allegations of e-waste mismanagement",
     "We are investigating allegations that e-waste collected through MobiCycle's programmes was exported to developing countries rather than being properly recycled. We would like your comment before publication.",
     {"complexity":"high","urgency":"immediate","topic_esg":"pollution","topic_operational":"client","entity_routing":"mobicycle.news","sentiment":"negative","formality":"business","response_type":"escalate","reasoning":"Media investigation about core business. Reputational emergency. Route to news division for communications. Requires legal review before any response."}),

    ("Customs", "enforcement@ukborderforce.gov.uk",
     "Detained shipment: suspected illegal waste export",
     "Container MSKU7654321 consigned to MobiCycle has been detained at Felixstowe port. Initial inspection suggests the contents may constitute illegal waste export under the Basel Convention.",
     {"complexity":"high","urgency":"immediate","topic_esg":"pollution","topic_operational":"legal","entity_routing":"same_account","sentiment":"negative","formality":"formal","response_type":"escalate","reasoning":"Customs detention of shipment. Potential Basel Convention violation — criminal offence. Pollution/waste core area. Requires immediate legal response."}),

    ("Former Employee", "solicitor@employmentlaw.co.uk",
     "Pre-action protocol letter: whistleblower detriment",
     "Our client was dismissed from MobiCycle after raising concerns about environmental compliance. This constitutes automatic unfair dismissal under s.103A ERA 1996. We invite resolution within 28 days.",
     {"complexity":"high","urgency":"time-sensitive","topic_esg":"pollution","topic_operational":"legal","entity_routing":"same_account","sentiment":"hostile","formality":"formal","response_type":"escalate","reasoning":"Whistleblower detriment claim. Automatic unfair dismissal — no qualifying period. Intersects pollution/compliance concerns. High risk. 28-day pre-action window."}),


    # ──── MEDIUM COMPLEXITY (40) ────
    ("Client Sustainability", "esg@retailclient.com",
     "Scope 3 data collection: categories 1-5 needed by month end",
     "Hi, we're compiling our annual CDP submission and need the upstream Scope 3 data for categories 1-5. Can you send the analysis by end of March? Happy to jump on a call if needed.",
     {"complexity":"medium","urgency":"time-sensitive","topic_esg":"scope-3","topic_operational":"client","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"Client requesting Scope 3 data for CDP. End-of-month deadline. Requires data compilation from existing analyses. Standard consulting deliverable."}),

    ("Partner Firm", "collab@enviro-consultants.com",
     "Joint bid for council waste audit contract",
     "The council tender for the waste management audit is due in 3 weeks. I've drafted the methodology section. Could you review and add the e-waste compliance section? Budget allocation needs discussion.",
     {"complexity":"medium","urgency":"time-sensitive","topic_esg":"pollution","topic_operational":"partnership","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"Joint tender with 3-week deadline. Requires methodology review and e-waste input. Budget discussion needed but no legal complexity."}),

    ("ISO Auditor", "certification@bsigroup.com",
     "ISO 14001 surveillance audit: non-conformities to address",
     "Following the surveillance audit on 5 March, two minor non-conformities were identified. Please submit evidence of corrective actions within 60 days to maintain certification.",
     {"complexity":"medium","urgency":"standard","topic_esg":"pollution","topic_operational":"client","entity_routing":"same_account","sentiment":"neutral","formality":"formal","response_type":"respond","reasoning":"ISO audit non-conformities — minor, with 60-day window. Requires corrective action evidence. Standard quality management process."}),

    ("New Client", "enquiry@techstartup.com",
     "RFP: carbon footprint assessment for Series B due diligence",
     "We're a climate tech startup raising Series B. Our investors require an independent carbon footprint assessment. Could you send a proposal including timeline and pricing? Budget is around £25-40K.",
     {"complexity":"medium","urgency":"standard","topic_esg":"scope-3","topic_operational":"client","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"New business opportunity. Scope 3 assessment for investor DD. Requires proposal drafting with pricing. £25-40K engagement."}),

    ("Accountant", "accounts@accountingfirm.co.uk",
     "Management accounts Q4: review and sign-off needed",
     "Please find attached the draft management accounts for Q4 2025. Revenue is up 12% but margins have declined due to increased subcontractor costs. Please review and confirm for board pack.",
     {"complexity":"medium","urgency":"time-sensitive","topic_esg":None,"topic_operational":"financial","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"respond","reasoning":"Quarterly management accounts requiring review and sign-off. Board pack deadline. Financial analysis needed on margin decline."}),

    ("GRI", "standards@globalreporting.org",
     "Updated GRI 304 Biodiversity standard: implementation guidance",
     "The revised GRI 304 standard on Biodiversity takes effect for reports published after January 2027. Enclosed is the implementation guidance. We recommend starting gap analysis now.",
     {"complexity":"medium","urgency":"standard","topic_esg":"biodiversity","topic_operational":"client","entity_routing":"same_account","sentiment":"neutral","formality":"formal","response_type":"respond","reasoning":"New biodiversity reporting standard. Affects client advisory. Not urgent but requires gap analysis. Relevance to multiple client engagements."}),

    ("Client Ops", "operations@logistics.com",
     "Supply chain mapping workshop: scheduling for next month",
     "We'd like to schedule the Scope 3 supply chain mapping workshops. Three sessions needed — procurement, logistics, and packaging teams. Can you propose dates for April?",
     {"complexity":"medium","urgency":"standard","topic_esg":"scope-3","topic_operational":"client","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"Client workshop scheduling. Scope 3 supply chain mapping. Requires diary coordination for three sessions. Standard project management."}),

    ("SBTi", "targets@sciencebasedtargets.org",
     "Target validation: additional data requested",
     "During validation of your client's SBTi near-term targets, we require additional data on fleet emissions methodology and office energy consumption calculations. Please respond within 30 days.",
     {"complexity":"medium","urgency":"standard","topic_esg":"scope-3","topic_operational":"client","entity_routing":"same_account","sentiment":"neutral","formality":"formal","response_type":"respond","reasoning":"SBTi data request during target validation. 30-day window. Requires retrieval of methodology documentation. Standard validation process."}),

    ("Internal PM", "projects@mobicycle.consulting",
     "Resource allocation: two new projects starting in April",
     "We've won two new engagements starting April. Need to discuss team allocation — the biodiversity assessment needs an ecologist, and the CSRD engagement needs someone with EU experience.",
     {"complexity":"medium","urgency":"standard","topic_esg":"biodiversity","topic_operational":"hr","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"Resource planning for new projects. Requires skills matching and availability checking. Biodiversity and EU expertise needed."}),

    ("Recruiter", "hire@talentfirm.com",
     "Shortlisted candidates: senior sustainability consultant",
     "Three candidates have been shortlisted for the senior sustainability consultant role. CVs attached. Two have Big 4 experience, one comes from industry. When would you like to schedule interviews?",
     {"complexity":"medium","urgency":"standard","topic_esg":None,"topic_operational":"hr","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"Recruitment decision with three shortlisted candidates. Requires CV review and interview scheduling. Standard HR process."}),

    ("Marketing Team", "marketing@mobicycle.marketing",
     "Case study draft: RetailCo Scope 3 project",
     "Hi, I've drafted a case study based on the RetailCo engagement. Can you review for technical accuracy? Also need client approval before publishing. Target publish date: mid-April.",
     {"complexity":"medium","urgency":"standard","topic_esg":"scope-3","topic_operational":"client","entity_routing":"same_account","sentiment":"positive","formality":"casual","response_type":"respond","reasoning":"Case study review. Needs technical accuracy check and client approval. Marketing coordination. Standard content review process."}),

    ("Conference Organiser", "speakers@esgworld.com",
     "Confirmed: keynote slot at ESG World 2026",
     "Great news — you've been confirmed for a 30-minute keynote at ESG World in Berlin, 15 May. Please submit your presentation title, abstract (200 words), and bio by 15 April.",
     {"complexity":"medium","urgency":"standard","topic_esg":"scope-3","topic_operational":"partnership","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"Conference keynote requiring abstract and bio submission. Good marketing opportunity. Mid-April deadline. Presentation preparation needed."}),

    ("IT Support", "helpdesk@mobicycle.tech",
     "Migration to new email platform: action required",
     "We're migrating from Spark to a new email platform next month. Each team member needs to export their local folders and configure the new client. Instructions attached. Please complete by 10 April.",
     {"complexity":"medium","urgency":"standard","topic_esg":None,"topic_operational":"infrastructure","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"respond","reasoning":"IT migration requiring team action. 10 April deadline. Operational disruption risk if not completed. Standard infrastructure change."}),

    ("Subcontractor", "availability@freelancer.com",
     "Availability check: WEEE audit next month",
     "Hi, are you available for a 3-day WEEE compliance audit at a manufacturing site in Birmingham, week of 14 April? Rate is £650/day. Please confirm by end of week.",
     {"complexity":"medium","urgency":"time-sensitive","topic_esg":"pollution","topic_operational":"client","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"Subcontractor availability for WEEE audit. End-of-week response needed. Budget and scheduling decision required."}),

    ("CIEEM", "events@cieem.net",
     "BNG workshop: practical implementation for consultants",
     "Biodiversity Net Gain is now mandatory for most developments. This 2-day workshop covers practical BNG assessment for consultants. Early bird: £450. Dates: 22-23 April.",
     {"complexity":"medium","urgency":"standard","topic_esg":"biodiversity","topic_operational":"hr","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"respond","reasoning":"Professional development opportunity in biodiversity. Relevant to service offering. Requires budget approval and scheduling decision."}),

    ("Client Compliance", "compliance@pharmagroup.com",
     "Annual sustainability report: data collection spreadsheet",
     "Attached is the data collection spreadsheet for our 2025 sustainability report. We need energy, water, waste, and emissions data for all sites. Deadline: 20 April. Call on Thursday to align?",
     {"complexity":"medium","urgency":"time-sensitive","topic_esg":"scope-3","topic_operational":"client","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"respond","reasoning":"Client sustainability report data collection. 20 April deadline. Multi-site data gathering. Standard annual process but time-bounded."}),

    ("Supplier", "orders@monitoringequipment.com",
     "Order confirmation: air quality monitors — delivery delayed",
     "Your order for 12 PM2.5 monitors has been confirmed but delivery is delayed by 2 weeks due to supply chain issues. New estimated delivery: 8 April. Apologies for the inconvenience.",
     {"complexity":"medium","urgency":"standard","topic_esg":"pollution","topic_operational":"infrastructure","entity_routing":"same_account","sentiment":"negative","formality":"business","response_type":"respond","reasoning":"Equipment delivery delay affecting project timeline. Pollution monitoring equipment. Need to assess impact on client commitments and consider alternatives."}),

    ("Fleet Manager", "vehicles@fleetco.com",
     "EV transition plan: lease options for company vehicles",
     "Three vehicle leases expire in June. I've prepared options for EV replacements including charging infrastructure. Total annual cost difference is +£4,200 but the Scope 1 reduction is significant.",
     {"complexity":"medium","urgency":"standard","topic_esg":"scope-3","topic_operational":"financial","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"respond","reasoning":"Fleet EV transition decision. Financial vs ESG trade-off. June deadline for lease expiry. Requires cost-benefit review."}),

    ("Ellen MacArthur Foundation", "circular@emf.org",
     "Circular economy network: annual membership renewal",
     "Your CE100 membership expires on 30 April. Renewal fee is £5,000. This year's programme includes exclusive workshops on plastic packaging circularity and electronics take-back.",
     {"complexity":"medium","urgency":"standard","topic_esg":"pollution","topic_operational":"partnership","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"respond","reasoning":"Membership renewal decision. £5K budget item. Relevant to circular economy and e-waste consulting. 30 April deadline."}),

    ("Property Agent", "commercial@estateagent.co.uk",
     "Office expansion: viewing arranged for 3 options",
     "Based on your requirements, I've shortlisted three office spaces in the area. Viewings are arranged for next Tuesday. Details and floor plans attached. All have EPC rating B or above.",
     {"complexity":"medium","urgency":"standard","topic_esg":None,"topic_operational":"infrastructure","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"Office space viewings requiring decision. Three options to evaluate. Tuesday viewing needs diary coordination. Infrastructure decision."}),

    ("Tööinspektsioon", "kontroll@ti.ee",
     "Töökeskkonna kontroll: dokumentide esitamine",
     "Palume esitada töökeskkonna riskianalüüs ja tervisekontrolli tulemused 30 päeva jooksul vastavalt töötervishoiu ja tööohutuse seaduse §-le 134.",
     {"complexity":"medium","urgency":"standard","topic_esg":None,"topic_operational":"legal","entity_routing":"mobicycle.ee","sentiment":"neutral","formality":"formal","response_type":"respond","reasoning":"Estonian workplace health inspection. 30-day document submission. Route to Estonian entity. Standard regulatory compliance."}),

    ("CDP", "disclose@cdp.net",
     "2026 Climate Change questionnaire: submission window open",
     "The 2026 CDP Climate Change questionnaire is now open. Submission deadline: 31 July. New modules on Scope 3 Category 15 (investments) and Nature-related disclosures have been added.",
     {"complexity":"medium","urgency":"standard","topic_esg":"scope-3","topic_operational":"client","entity_routing":"same_account","sentiment":"neutral","formality":"formal","response_type":"respond","reasoning":"Annual CDP submission window. July deadline gives time. New modules need review. Affects multiple client engagements."}),

    ("Research Partner", "collab@university.ac.uk",
     "Joint research proposal: circular economy metrics",
     "Following our discussion, I've drafted the UKRI funding application for the circular economy metrics research. Your section on industry application needs completing. Application deadline: 1 May.",
     {"complexity":"medium","urgency":"time-sensitive","topic_esg":"pollution","topic_operational":"partnership","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"Research funding application with 1 May deadline. Requires section completion. Partnership coordination. Circular economy topic."}),

    ("Internal Finance", "finance@mobicycle.consulting",
     "Invoice queries: 3 outstanding client invoices",
     "Three invoices totalling £42,300 are more than 45 days overdue. RetailCo (£18K), GreenTech (£15K), and PharmaGroup (£9.3K). Can you follow up with client contacts?",
     {"complexity":"medium","urgency":"time-sensitive","topic_esg":None,"topic_operational":"financial","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"respond","reasoning":"Overdue invoices totalling £42.3K. Cash flow impact. Requires client relationship management to chase payment. Three separate follow-ups needed."}),

    ("Cloud Provider", "enterprise@aws.com",
     "Annual contract review: usage up 45%, renewal terms",
     "Your annual cloud spend has increased 45% YoY. Current contract expires June 30. We'd like to propose a 3-year commitment with 20% discount. Meeting to discuss?",
     {"complexity":"medium","urgency":"standard","topic_esg":None,"topic_operational":"infrastructure","entity_routing":"mobicycle.tech","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"Cloud contract renewal with cost implications. 45% usage increase to understand. Tech division should review infrastructure needs. June deadline."}),

    ("Standards Body", "certification@ukas.com",
     "UKAS accreditation: annual review documentation",
     "Your UKAS accreditation annual review is scheduled for 12 May. Please submit the required documentation including updated quality manual, staff competency records, and audit trail by 28 April.",
     {"complexity":"medium","urgency":"standard","topic_esg":None,"topic_operational":"client","entity_routing":"same_account","sentiment":"neutral","formality":"formal","response_type":"respond","reasoning":"Accreditation review requiring document preparation. 28 April document deadline. Quality management process. Important for business credibility."}),

    ("Insurance Broker", "renewal@broker.co.uk",
     "Annual insurance review: premium increase notification",
     "Following the market review, your combined insurance premium is increasing by 18% to £24,600. Main driver is the PI claim history. Alternative quotes attached. Decision needed by 15 April.",
     {"complexity":"medium","urgency":"time-sensitive","topic_esg":None,"topic_operational":"financial","entity_routing":"same_account","sentiment":"negative","formality":"business","response_type":"respond","reasoning":"Insurance premium increase of 18%. Alternative quotes to compare. 15 April decision deadline. Financial review and risk assessment needed."}),

    ("Client TNFD", "sustainability@miningco.com",
     "TNFD pilot: location-specific biodiversity assessment",
     "We'd like to pilot the TNFD LEAP approach at three of our mining sites. Can you scope the work? We need a proposal covering methodology, timeline, and pricing by end of April.",
     {"complexity":"medium","urgency":"standard","topic_esg":"biodiversity","topic_operational":"client","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"New business opportunity for TNFD biodiversity assessment. Three-site scope. Proposal preparation needed. End of April deadline."}),

    ("Charity Partner", "partnership@ocean-conservation.org",
     "Corporate sponsorship: ocean plastic awareness campaign",
     "We're launching a major ocean plastic awareness campaign in June. Would MobiCycle be interested in being a headline sponsor? Packages range from £5K to £25K. Great alignment with your pollution focus.",
     {"complexity":"medium","urgency":"standard","topic_esg":"pollution","topic_operational":"partnership","entity_routing":"mobicycle.marketing","sentiment":"positive","formality":"business","response_type":"respond","reasoning":"Sponsorship opportunity aligned with pollution mission. Budget decision needed. Route to marketing for comms alignment. No urgent deadline."}),

    ("Software Vendor", "support@esg-platform.com",
     "Platform update: breaking changes to API v2",
     "API v2 will be deprecated on 1 June. You must migrate to v3 before then. Key changes: new auth flow, different data schema for emissions factors, and rate limit changes. Migration guide attached.",
     {"complexity":"medium","urgency":"standard","topic_esg":"scope-3","topic_operational":"infrastructure","entity_routing":"mobicycle.tech","sentiment":"neutral","formality":"business","response_type":"respond","reasoning":"API migration with 1 June deadline. Breaking changes affecting emissions calculations. Route to tech for implementation. Plan migration timeline."}),

    ("Internal Training", "learning@mobicycle.consulting",
     "Mandatory training: anti-bribery and corruption refresher",
     "All staff must complete the annual ABC refresher by 30 April. The online module takes approximately 45 minutes. Completion certificates will be tracked for compliance records.",
     {"complexity":"medium","urgency":"standard","topic_esg":None,"topic_operational":"hr","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"respond","reasoning":"Mandatory compliance training with 30 April deadline. Requires scheduling 45 minutes. Standard annual requirement."}),

    ("Waste Processor", "contracts@wastemanagement.com",
     "Contract variation: new WEEE categories and pricing",
     "Due to changes in the WEEE Regulations, we need to add three new waste categories to our processing contract. Revised pricing schedule attached. Please review and sign the variation by 15 April.",
     {"complexity":"medium","urgency":"time-sensitive","topic_esg":"pollution","topic_operational":"client","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"respond","reasoning":"Contract variation for WEEE compliance. New pricing to review. 15 April signature deadline. E-waste regulatory change driving update."}),


    # ──── LOW COMPLEXITY (40) ────
    ("LinkedIn", "notifications@linkedin.com",
     "5 people viewed your profile this week",
     "See who's been looking at your profile. Your profile appeared in 42 searches this week. Top industries: Environmental Consulting, Corporate Sustainability.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"casual","response_type":"file","reasoning":"Automated LinkedIn notification. No response needed. File for reference."}),

    ("Conference", "register@climateweek.com",
     "Climate Week NYC 2026: early bird registration",
     "Climate Week NYC returns 21-27 September 2026. Early bird tickets available until 1 June. Sessions on nature-based solutions, carbon markets, and corporate net zero.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"scope-3","topic_operational":None,"entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"file","reasoning":"Conference promotion. September event. No action needed now. File for future consideration."}),

    ("Industry Report", "research@mckinsey.com",
     "New report: The State of Sustainability 2026",
     "Our latest research on corporate sustainability reveals that 78% of companies are now setting science-based targets, up from 62% in 2024. Download the full report.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"scope-3","topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Industry report for reference. No response required. Useful background data on SBTi adoption rates."}),

    ("Cloudflare", "noreply@notify.cloudflare.com",
     "Certificate transparency monitoring for mobicycle.consulting",
     "A new SSL/TLS certificate was issued for mobicycle.consulting by Let's Encrypt. This is expected if you recently renewed your certificate.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":"infrastructure","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Automated certificate notification. Expected renewal. No action needed. File as infrastructure record."}),

    ("Newsletter", "digest@edie.net",
     "edie Weekly: latest sustainability news",
     "This week: EU Green Deal updates, new ISSB standards timeline, UK Emissions Trading Scheme expansion, and corporate greenwashing crackdown.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"scope-3","topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Industry newsletter. Background reading. No response required."}),

    ("Colleague", "mike@mobicycle.consulting",
     "Coffee run: want anything?",
     "Heading to the coffee shop. The usual?",
     {"complexity":"low","urgency":"immediate","topic_esg":None,"topic_operational":None,"entity_routing":"same_account","sentiment":"positive","formality":"casual","response_type":"acknowledge","reasoning":"Informal colleague message. Quick acknowledgment needed. No business substance."}),

    ("Alumni Network", "events@alumni-oxford.ac.uk",
     "Sustainability Alumni Network: summer gathering",
     "Join fellow alumni working in sustainability for drinks and networking at the Oxford Union on 12 June. RSVP by 1 June.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":None,"entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"file","reasoning":"Networking event invitation. Optional attendance. No business urgency. File for calendar consideration."}),

    ("Spark", "support@readdle.com",
     "Your Spark subscription has been renewed",
     "Your Spark Premium subscription has been successfully renewed for another year. Amount charged: £49.99.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":"financial","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Subscription renewal confirmation. No action needed. File as financial record."}),

    ("Podcast", "booking@sustainabilitypodcast.com",
     "Guest invitation: share your circular economy expertise",
     "Would you be interested in being a guest on our podcast to discuss e-waste challenges in the consulting industry? Episodes are 30 minutes. We can record at your convenience.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"pollution","topic_operational":"partnership","entity_routing":"same_account","sentiment":"positive","formality":"casual","response_type":"respond","reasoning":"Podcast guest invitation. Good marketing opportunity. No deadline. Low effort to accept or decline."}),

    ("Former Colleague", "hello@ex-colleague.com",
     "Catching up: moved to a new role",
     "Hi! Just wanted to let you know I've moved to GreenTech Corp as Head of Sustainability. Would be great to catch up sometime. Coffee or a call?",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":"partnership","entity_routing":"same_account","sentiment":"positive","formality":"casual","response_type":"respond","reasoning":"Personal networking. Former colleague in relevant role. Potential future business. Respond at convenience."}),

    ("Office Supplies", "orders@eco-stationery.com",
     "New recycled paper range now available",
     "Our new 100% post-consumer recycled paper range is now in stock. 15% introductory discount for existing customers. Order before April 30.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"pollution","topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Supplier promotion. No urgent need. File for when supplies are needed."}),

    ("Google Calendar", "calendar-notification@google.com",
     "Reminder: team standup tomorrow at 9:30",
     "Team standup - Tomorrow 9:30-9:45 AM. Google Meet link attached.",
     {"complexity":"low","urgency":"standard","topic_esg":None,"topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Automated calendar reminder. No response needed. Already in calendar."}),

    ("Climate Pledge", "updates@climatepledge.com",
     "Annual signatory progress review: submit by September",
     "As a Climate Pledge signatory, please submit your annual progress report by 30 September 2026. Template attached.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"scope-3","topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Annual reporting requirement with September deadline. Very distant. File for future action."}),

    ("Team", "team@mobicycle.consulting",
     "Friday social: pub quiz this Friday",
     "We're organising a pub quiz at The Green Man this Friday at 6pm. Teams of 4. Let me know if you're in!",
     {"complexity":"low","urgency":"standard","topic_esg":None,"topic_operational":None,"entity_routing":"same_account","sentiment":"positive","formality":"casual","response_type":"acknowledge","reasoning":"Social event. Quick yes/no response. No business substance."}),

    ("GitHub", "noreply@github.com",
     "Security alert: dependabot found 2 vulnerabilities",
     "Dependabot found 2 moderate severity vulnerabilities in your repository mobicycle/cf-email. Automated pull requests have been created.",
     {"complexity":"low","urgency":"standard","topic_esg":None,"topic_operational":"infrastructure","entity_routing":"mobicycle.tech","sentiment":"neutral","formality":"business","response_type":"forward","reasoning":"Automated security alert. Moderate severity. Auto-PRs created. Route to tech division for merge review."}),

    ("Webinar", "events@bsigroup.com",
     "Free webinar: ISO 14064 greenhouse gas verification",
     "Join our free 1-hour webinar on ISO 14064 greenhouse gas verification on 18 April at 2pm. Topics: verification methodology, common pitfalls, and regulatory requirements.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"scope-3","topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Free webinar on relevant topic. Optional attendance. File for consideration."}),

    ("Stripe", "receipts@stripe.com",
     "Payment receipt: £299.00 to Figma",
     "Your payment of £299.00 to Figma Inc has been processed. Transaction ID: pi_3QwXyz123. Receipt attached.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":"financial","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Payment receipt. No action needed. File for accounting records."}),

    ("Client Thank You", "manager@satisfied-client.com",
     "Thank you for the excellent sustainability report",
     "Just wanted to say thank you for the brilliant work on our sustainability report. The board was very impressed. Looking forward to working together again.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":"client","entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"acknowledge","reasoning":"Client appreciation. Acknowledge with thanks. Good for case study material. No action required."}),

    ("Mentor", "personal@industry-veteran.com",
     "Interesting article on biodiversity credits",
     "Saw this article on the emerging biodiversity credit market. Thought you'd find it interesting given your TNFD work. Link: [article URL]",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"biodiversity","topic_operational":None,"entity_routing":"same_account","sentiment":"positive","formality":"casual","response_type":"acknowledge","reasoning":"Shared article from mentor. Thank and note the reference. Biodiversity credit market relevant to advisory work."}),

    ("Award Entry", "entries@businessawards.com",
     "Nomination open: Sustainability Consultancy of the Year",
     "Nominations are now open for the 2026 Business Green Awards. Category: Sustainability Consultancy of the Year. Entry deadline: 31 May. Entry fee: £250.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":None,"entity_routing":"mobicycle.marketing","sentiment":"positive","formality":"business","response_type":"forward","reasoning":"Award nomination opportunity. Route to marketing for decision on entry. May deadline. £250 entry fee."}),

    ("Wellness", "corporate@mindfulness-app.com",
     "Corporate wellness: free 30-day trial",
     "Offer your team free access to our mindfulness and stress management app. Over 500 companies use us. Set up a free 30-day trial.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":"hr","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Promotional email. Wellness offering. No immediate need. File or ignore."}),

    ("GitHub", "noreply@github.com",
     "[mobicycle/cf-www] Pull request merged",
     "Pull request #47 'Update footer links' has been merged into main by @mobicycle-bot.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":"infrastructure","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Automated GitHub notification. PR merged successfully. No action needed."}),

    ("PhD Student", "research@university-phd.ac.uk",
     "Research survey: consulting industry carbon footprint",
     "Hi, I'm a PhD student researching the carbon footprint of the consulting industry. Would you be willing to complete a 10-minute survey? All responses are anonymous.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"scope-3","topic_operational":None,"entity_routing":"same_account","sentiment":"positive","formality":"casual","response_type":"respond","reasoning":"Academic survey request. 10 minutes. Optional. Good university relations. Respond at convenience."}),

    ("Gym", "corporate@fitness-club.com",
     "Corporate membership: 25% discount for your team",
     "Exclusive corporate rate: £45/month per person (normally £60). Includes all classes and gym access at any branch.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":"hr","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Promotional email for gym membership. Not relevant to operations. File or ignore."}),

    ("Proton Mail", "no-reply@protonmail.com",
     "Your Proton Mail storage is 75% full",
     "You're using 11.25 GB of your 15 GB storage. Consider upgrading or clearing old emails.",
     {"complexity":"low","urgency":"standard","topic_esg":None,"topic_operational":"infrastructure","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"acknowledge","reasoning":"Storage notification. Not urgent at 75%. May need action later. Note for housekeeping."}),

    ("Book Author", "author@sustainability-book.com",
     "Advance copy: The Circular Economy Handbook 2nd Edition",
     "As a contributor, here's your advance copy of the 2nd edition. Your chapter on e-waste consulting is on page 234. The book launches 1 May.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"pollution","topic_operational":None,"entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"acknowledge","reasoning":"Book contributor copy. Acknowledge receipt. Good for credibility. No action required beyond thanks."}),

    ("Printer", "orders@printshop.com",
     "Business cards: reorder reminder",
     "Your last order of 500 business cards was 6 months ago. Time for a reorder? Same design or updated version? 10% reorder discount.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Automated reorder reminder. Not urgent. File for when stock runs low."}),

    ("Pension Provider", "statements@pensionco.com",
     "Annual pension statement 2025/26",
     "Your annual pension statement is ready. Current fund value: £42,156. Annual contribution: £4,800. Projected retirement income: see attached.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":"financial","entity_routing":"same_account","sentiment":"neutral","formality":"formal","response_type":"file","reasoning":"Annual pension statement. Informational only. File for personal records. No response needed."}),

    ("Office Plants", "service@officeplants.co.uk",
     "Monthly plant maintenance: visit confirmed for Thursday",
     "Your monthly plant maintenance visit is confirmed for Thursday 27 March, 10am-11am. Our technician will water, prune, and replace any plants as needed.",
     {"complexity":"low","urgency":"standard","topic_esg":None,"topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Routine service confirmation. No action needed. Already scheduled."}),

    ("CSR Newsletter", "newsletter@mobicycle.us",
     "Monthly update: what's happening across MobiCycle",
     "This month: new WEEE directive compliance tool launched by Games division, EU office expansion on track, three new client wins in Consulting.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":None,"entity_routing":"same_account","sentiment":"positive","formality":"business","response_type":"file","reasoning":"Internal company newsletter. Background reading. No response needed."}),

    ("Travel Agent", "bookings@businesstravel.com",
     "Flight confirmation: London to Berlin, 14 May",
     "Your flight BA984 LHR-BER on 14 May is confirmed. Departure 07:15, arrival 10:10. E-ticket attached. Carbon offset: 0.18 tonnes CO2.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"scope-3","topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Flight booking confirmation. No action needed. File for travel records and carbon tracking."}),

    ("IEMA", "members@iema.net",
     "New IEMA publication: Environmental Management Best Practice",
     "Free for members: our latest publication on environmental management best practice. Covers ISO 14001:2015, lifecycle assessment, and materiality analysis.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"scope-3","topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Professional body publication. Reference material. No response required."}),

    ("IT Notification", "system@mobicycle.tech",
     "Scheduled maintenance: server downtime Sunday 2am-4am",
     "Planned maintenance window: Sunday 30 March, 02:00-04:00 UTC. All services will be briefly unavailable. No action required from you.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":None,"topic_operational":"infrastructure","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"file","reasoning":"Planned maintenance notification. No action needed. File for awareness."}),

    ("Xero", "noreply@xero.com",
     "Bank reconciliation: 3 items need attention",
     "3 bank transactions have not been reconciled in your MobiCycle Consulting account. Please review at your convenience.",
     {"complexity":"low","urgency":"standard","topic_esg":None,"topic_operational":"financial","entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"respond","reasoning":"Bookkeeping notification. 3 items to reconcile. Standard accounting task. No urgency."}),

    ("Thought Leader", "share@esg-influencer.com",
     "New blog: Why biodiversity is the next carbon",
     "My latest blog explores why biodiversity disclosure is following the same trajectory as carbon reporting did 10 years ago. Share with your network if you agree.",
     {"complexity":"low","urgency":"no-deadline","topic_esg":"biodiversity","topic_operational":None,"entity_routing":"same_account","sentiment":"positive","formality":"casual","response_type":"file","reasoning":"Thought leadership content. Biodiversity topic relevant. Share or file for reference."}),

    ("Cleaning Company", "service@cleanoffice.co.uk",
     "Deep clean scheduled: Friday 28 March after 6pm",
     "Your quarterly deep clean is scheduled for Friday evening. Please ensure desks are cleared. Any areas requiring special attention?",
     {"complexity":"low","urgency":"standard","topic_esg":None,"topic_operational":None,"entity_routing":"same_account","sentiment":"neutral","formality":"business","response_type":"acknowledge","reasoning":"Scheduled cleaning confirmation. Quick acknowledgment if special areas needed. Otherwise no action."}),
]

# Generate train-assess.csv
random.seed(42)
random.shuffle(EXAMPLES)
os.makedirs('/content/data/assess', exist_ok=True)
train_path = '/content/data/assess/train-assess.csv'
with open(train_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['text'])
    for name, email, subject, body, assessment in EXAMPLES:
        # Convert None to null string for JSON
        assess_str = json.dumps(assessment)
        prompt = (f'<start_of_turn>user\n'
                  f'Assess this email and return a JSON assessment with fields: '
                  f'complexity, urgency, topic_esg, topic_operational, entity_routing, '
                  f'sentiment, formality, response_type, reasoning.\n\n'
                  f'Account: {ACCOUNT}\n'
                  f'From: {name} <{email}>\n'
                  f'Subject: {subject}\n'
                  f'Body: {body}\n'
                  f'<end_of_turn>\n'
                  f'<start_of_turn>model\n'
                  f'{assess_str}<end_of_turn>')
        w.writerow([prompt])

# Verify distribution
from collections import Counter
complexity_counts = Counter(a['complexity'] for *_, a in EXAMPLES)
urgency_counts = Counter(a['urgency'] for *_, a in EXAMPLES)
response_counts = Counter(a['response_type'] for *_, a in EXAMPLES)
print(f'Total: {len(EXAMPLES)} examples')
print(f'\nComplexity: {dict(complexity_counts)}')
print(f'Urgency: {dict(urgency_counts)}')
print(f'Response type: {dict(response_counts)}')
print(f'\nSaved to {train_path}')

In [ ]:
# ── Cell 4: Train QLoRA ──
import sys, types, json, os, math

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))
    print('Config reloaded')

# Fix Colab triton bug
try:
    import triton
    if not hasattr(triton, 'ops'):
        triton.ops = types.ModuleType('triton.ops')
        sys.modules['triton.ops'] = triton.ops
except ImportError:
    pass

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
import torch

# ── Load training data from Cell 3 ──
def format_example(ex):
    from_name, from_email, subject, body, assessment = ex
    prompt = f"""Assess this email and return a JSON object.

From: {from_name} <{from_email}>
Subject: {subject}
Body: {body}"""
    response = json.dumps(assessment, ensure_ascii=False)
    return {"text": f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n{response}<end_of_turn>"}

dataset = Dataset.from_list([format_example(ex) for ex in EXAMPLES])
print(f'Training examples: {len(dataset)}')

# ── 4-bit quantization (aggressive for 12B on T4) ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # Double quant saves ~0.4GB
)

# ── Load tokenizer and model ──
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config, device_map={'': 0},
    trust_remote_code=True, dtype=torch.float16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
print(f'Model loaded: {MODEL} (4-bit double-quant)')

# ── LoRA config ──
# Target attention layers — Gemma 3 uses same naming as Gemma 2
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Training ──
project = f'{ADAPTER_PREFIX}-{ACCOUNT}'
output_dir = f'/content/{project}'

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=5,
    save_strategy='epoch',
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    gradient_checkpointing=True,  # Essential for 12B on T4
    optim='paged_adamw_8bit',  # 8-bit optimizer saves VRAM
    max_grad_norm=0.3,
    report_to='none',
)

# ── Patch Gemma 3 for text-only training (no vision tokens) ──
import transformers.models.gemma3.modeling_gemma3 as g3mod

_orig_create_mask = g3mod.create_causal_mask_mapping

def _text_only_mask(config, input_embeds, attention_mask=None, cache_position=None,
                    past_key_values=None, position_ids=None, token_type_ids=None,
                    pixel_values=None, is_training=False, is_first_iteration=True, **kwargs):
    batch_size, seq_len = input_embeds.shape[:2]
    token_type_ids = torch.zeros(batch_size, seq_len, dtype=torch.long, device=input_embeds.device)
    return _orig_create_mask(
        config, input_embeds, attention_mask, cache_position, past_key_values,
        position_ids, token_type_ids=token_type_ids, pixel_values=pixel_values,
        is_training=is_training, is_first_iteration=is_first_iteration, **kwargs
    )

g3mod.create_causal_mask_mapping = _text_only_mask
print('Patched Gemma 3 for text-only training')

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

print(f'Starting training 12B on T4...')
trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'Adapter saved to {output_dir}')


In [ ]:
# ── Cell 5: Patch adapter_config.json for Workers AI ──
import json, os

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

project = f'{ADAPTER_PREFIX}-{ACCOUNT}'
output_dir = f'/content/{project}'
config_path = os.path.join(output_dir, 'adapter_config.json')

with open(config_path) as f:
    config = json.load(f)
config['model_type'] = MODEL_TYPE
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

for name in ['adapter_model.safetensors', 'adapter_config.json']:
    path = os.path.join(output_dir, name)
    if os.path.exists(path):
        print(f'{name}: {os.path.getsize(path) / 1e6:.1f} MB')
    else:
        print(f'{name}: MISSING!')
print(f'r={config.get("r")}, alpha={config.get("lora_alpha")}, model_type={config.get("model_type")}')

In [ ]:
# ── Cell 6: Save Locally (Base64 extraction) ──
# MANDATORY: Save adapter files BEFORE deploying.
import base64, os, json

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

project = f'{ADAPTER_PREFIX}-{ACCOUNT}'
output_dir = f'/content/{project}'

# Base64 encode safetensors in chunks (Colab output limit ~1MB)
CHUNK_SIZE = 750_000  # ~1MB base64 per chunk
safetensors_path = os.path.join(output_dir, 'adapter_model.safetensors')
config_path = os.path.join(output_dir, 'adapter_config.json')

with open(safetensors_path, 'rb') as f:
    data = f.read()
b64 = base64.b64encode(data).decode()
chunks = [b64[i:i+CHUNK_SIZE] for i in range(0, len(b64), CHUNK_SIZE)]

print(f'=== SAFETENSORS: {len(data)} bytes, {len(chunks)} chunks ===')
for i, chunk in enumerate(chunks):
    print(f'CHUNK_{i}_START')
    print(chunk)
    print(f'CHUNK_{i}_END')

# Config is small, single chunk
with open(config_path, 'r') as f:
    config_text = f.read()
print(f'=== CONFIG ===')
print(config_text)
print(f'=== END ===')
print(f'\nLocal save path: models/fine-tuning/PEFT/LoRA/assess/{ACCOUNT}/adapter/')

In [ ]:
# ── Cell 7: Deploy to Workers AI via cf-adapters ──
import requests, json, os, time

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

project = f'{ADAPTER_PREFIX}-{ACCOUNT}'
output_dir = f'/content/{project}'

# cf-adapters worker URLs per account
ADAPTERS_URLS = {
    'mobicycle.us': 'https://adapters.mobicycle.workers.dev',
    'mobicycle.consulting': 'https://adapters.mobicycle-consulting.workers.dev',
    'mobicycle.eu': 'https://adapters.mobicycle-eu.workers.dev',
    'mobicycle.ee': 'https://adapters.mobicycle-ou.workers.dev',
    'mobicycle.productions': 'https://adapters.mobicycle-productions.workers.dev',
    'mobicycle.games': 'https://adapters.mobicycle-games.workers.dev',
    'mobicycle.tech': 'https://adapters.mobicycle-tech.workers.dev',
    'mobicycle.marketing': 'https://adapters.mobicycle-marketing.workers.dev',
    'mobicycle.capital': 'https://adapters.mobicycle-capital.workers.dev',
    'mobicycle.support': 'https://adapters.mobicycle-support.workers.dev',
    'mobicycle.news': 'https://adapters.mobicycle-news.workers.dev',
}

base_url = ADAPTERS_URLS.get(ACCOUNT)
if not base_url:
    raise ValueError(f'No adapters URL for {ACCOUNT}')

adapter_name = project
print(f'Deploying {adapter_name} to {base_url}')

# Step 1: Check if broken entry exists
status = requests.get(f'{base_url}/api/status').json()
print(f'Current adapters: {json.dumps(status, indent=2)[:500]}')

# Step 2: Delete existing entry if present
adapters = requests.get(f'{base_url}/api/adapters').json()
for a in adapters:
    if a.get('name') == adapter_name or a.get('description', '').startswith(f'Stage 2 assessment'):
        print(f'Deleting existing entry: {a["id"]}')
        requests.delete(f'{base_url}/api/adapters/{a["id"]}')
        time.sleep(2)

# Step 3: Create new finetune entry
create_resp = requests.post(f'{base_url}/api/adapters', json={
    'name': adapter_name,
    'model': WORKERS_AI_MODEL,
    'description': f'Stage 2 assessment LoRA for {ACCOUNT}'
})
create_data = create_resp.json()
finetune_id = create_data.get('id') or create_data.get('finetune', {}).get('id')
print(f'Created finetune: {finetune_id}')

# Step 4: Upload adapter files
for filename in ['adapter_model.safetensors', 'adapter_config.json']:
    filepath = os.path.join(output_dir, filename)
    with open(filepath, 'rb') as f:
        upload_resp = requests.post(
            f'{base_url}/api/adapters/{finetune_id}/upload',
            files={'file': (filename, f, 'application/octet-stream')},
            data={'file_name': filename}
        )
    print(f'Uploaded {filename}: {upload_resp.status_code}')

print(f'\nDeployment complete. Wait 5s then verify...')
time.sleep(5)

In [ ]:
# ── Cell 8: Verify with Inference ──
# MANDATORY: Test the deployed adapter actually works.
import requests, json

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

# cf-email worker URLs per account
CF_EMAIL_URLS = {
    'mobicycle.us': 'https://cf-email.mobicycle.us',
    'mobicycle.consulting': 'https://cf-email.mobicycle.consulting',
    'mobicycle.eu': 'https://cf-email.mobicycle.eu',
    'mobicycle.ee': 'https://cf-email.mobicycle.ee',
    'mobicycle.productions': 'https://cf-email.mobicycle.productions',
    'mobicycle.games': 'https://cf-email.mobicycle.games',
    'mobicycle.tech': 'https://cf-email.mobicycle.tech',
    'mobicycle.marketing': 'https://cf-email.mobicycle.marketing',
    'mobicycle.capital': 'https://cf-email.mobicycle.capital',
    'mobicycle.support': 'https://cf-email.mobicycle.support',
    'mobicycle.news': 'https://cf-email.mobicycle.news',
}

base_url = CF_EMAIL_URLS.get(ACCOUNT)
if not base_url:
    raise ValueError(f'No cf-email URL for {ACCOUNT}')

print(f'Testing assessment on {ACCOUNT}...')

# Test with the /api/assess endpoint
resp = requests.post(f'{base_url}/api/assess', json={'limit': 1})
result = resp.json()

if resp.status_code == 200:
    print(f'SUCCESS: Assessment returned')
    print(json.dumps(result, indent=2)[:1000])
elif 'error' in result and '3037' in str(result):
    print(f'FAILED: Error 3037 — adapter files not uploaded correctly.')
    print(f'Delete the broken entry and re-upload from local save.')
    print(f'DO NOT proceed to next account.')
else:
    print(f'Status: {resp.status_code}')
    print(json.dumps(result, indent=2)[:500])
    print(f'\nIf /api/assess does not exist yet, test manually:')
    print(f'  curl -X POST {base_url}/api/assess -d \'{{"limit":1}}\'')